# Crypto swap-dodge test (#006 momentum)

Does closing before the daily rollover (flat ~minutes) and reopening dodge the FTMO 30%/side crypto swap?
Daily-bar cost-config comparison: **hold-through (pay swap)** vs **close+reopen (pay daily round-trip churn, no swap)** vs **swap-free ceiling**.
Base alpha = #006 crypto 30d XS momentum. PIT research universe (top-30 by trailing 30d $-volume) + FTMO-deployable subset reported separately.

In [1]:
# Cell 1 - Setup + data load + PIT research universe
import pandas as pd, numpy as np
from backtest.data import DataPanel

raw = pd.read_parquet("data/binance_hourly_top50_pool.parquet")
STABLE = {"BFUSDUSDT","RLUSDUSDT","USD1USDT","USDEUSDT","USDSUSDT","XAUTUSDT","PAXGUSDT"}
raw = raw[~raw.symbol.isin(STABLE)].copy()
raw["date"] = raw.open_time.dt.tz_convert("UTC").dt.normalize().dt.tz_localize(None)

# daily close (last hour) and daily $-volume (sum of quote_volume)
daily_close = raw.groupby(["date","symbol"])["close"].last().unstack("symbol").sort_index()
daily_qvol  = raw.groupby(["date","symbol"])["quote_volume"].sum().unstack("symbol").reindex_like(daily_close)
# base-unit volume for engine ADV (engine computes ADV = volume*price; quote_volume is already $)
daily_basevol = daily_qvol / daily_close

prices = daily_close
volume = daily_basevol

# PIT research universe: top-30 by trailing-30d $-volume, price must exist at t (info <= t)
trail = daily_qvol.rolling(30, min_periods=10).sum()
vol_rank = trail.rank(axis=1, ascending=False, method="first")
research_mask = ((vol_rank <= 30) & prices.notna()).reindex_like(prices).fillna(False)

panel = DataPanel(prices, volume=volume, universe=research_mask, check_outliers=True)

print(f"Bars: {len(panel.dates)}")
print(f"Assets (all): {len(panel.assets_all)}")
print(f"Range: {panel.dates[0].date()} -> {panel.dates[-1].date()}")
print(f"NaN fraction in prices: {prices.isna().mean().mean():.2%}")
print(f"Mean coins/day in research mask: {research_mask.sum(axis=1).mean():.1f}")
print(f"Mask coins/day range: {int(research_mask.sum(axis=1).min())}..{int(research_mask.sum(axis=1).max())}")
prices.tail(3).iloc[:, :5]

Bars: 2698
Assets (all): 59
Range: 2019-01-01 -> 2026-05-21
NaN fraction in prices: 40.37%
Mean coins/day in research mask: 26.6
Mask coins/day range: 0..30


C:\Users\User\backtest_engine\backtest\data\panel.py:172: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = self._prices.pct_change()
C:\Users\User\AppData\Local\Temp\ipykernel_30336\1271136780.py:24: UserWarning: DataPanel: 492 bar(s) with |return - median| > 10.0*MAD detected.
  Top examples:
    2021-01-28 DOGEUSDT: ret=+392.41%, mad_score=179.0
    2021-07-24 DEXEUSDT: ret=+172.35%, mad_score=77.0
    2024-12-03 TRXUSDT: ret=+96.11%, mad_score=65.8
    2021-04-16 DOGEUSDT: ret=+100.40%, mad_score=45.8
    2021-02-19 BNBUSDT: ret=+70.30%, mad_score=40.3
  Verify these are real moves, not bad ticks. Pass check_outliers=False to silence.
  panel = DataPanel(prices, volume=volume, universe=research_mask, check_outliers=True)


symbol,AAVEUSDT,ADAUSDT,ALGOUSDT,APTUSDT,ARBUSDT
date,,,,,
2026-05-19,87.43,0.2496,0.1127,0.93,0.1135
2026-05-20,88.06,NaN,0.1176,0.95,0.1101
2026-05-21,88.17,NaN,0.1153,0.95,0.1090


In [2]:
# Cell 2 - Strategy: #006 crypto 30d XS momentum (alpha pipeline -> PrecomputedAlpha)
from backtest.alpha_pipeline import signal_construction as sc, rank, neutralisation, decay
from backtest.strategy import Strategy
from backtest.risk import RiskConfig

# #006 pipeline, masked to the PIT top-30 universe so rank/neutralise span only tradable coins
sig   = sc.run(prices, "momentum", lookback=30)        # +sum(log returns), 30d formation
sig_m = sig.where(research_mask)                        # NaN outside top-30
r     = rank.run(sig_m)                                 # winsor 5/95 -> [-1,1]
n1    = neutralisation.run(r, "market")                 # dollar-neutral demean
d     = decay.run(n1, "linear", window=5)               # linear decay 5 (wts 5..1)
n2    = neutralisation.run(d, "market")                 # re-neutralise after decay
final = n2.shift(1)                                     # delay-1 (trade on prior-close info)
gross = final.abs().sum(axis=1).replace(0, np.nan)
weights = final.div(gross, axis=0).fillna(0.0)          # gross leverage = 1.0 / bar

class PrecomputedAlpha(Strategy):
    """#006-style precomputed daily weight frame; same class reused across cost configs."""
    rebalance_frequency = "daily"
    risk = RiskConfig(max_position=0.15, max_gross=1.0)
    def __init__(self, weights, tag="mom30"):
        self._w = weights; self.tag = tag
    def __repr__(self): return f"PrecomputedAlpha(tag={self.tag})"
    def required_data(self): return {"prices": None, "volume": None}
    def generate_weights(self, data, t):
        if t not in self._w.index:
            return pd.Series(0.0, index=data.assets)
        return self._w.loc[t].reindex(data.assets).fillna(0.0)

strat = PrecomputedAlpha(weights)
active = weights.abs().sum(axis=1) > 0
print("strat:", repr(strat))
print("rebalances/yr:", round(len(strat.rebalance_dates(panel.dates))/(len(panel.dates)/365),1))
print("first active bar:", weights.index[active.argmax()].date())
print("per-bar GROSS (active) mean:", round(float(weights[active].abs().sum(axis=1).mean()),4))
print("per-bar NET (active) max |sum|:", float(weights[active].sum(axis=1).abs().max()))
print("max single |raw weight| (pre risk-cap):", round(float(weights.abs().max().max()),4))
print("mean #long / #short per active bar:",
      round(float((weights>0).sum(axis=1)[active].mean()),1), "/",
      round(float((weights<0).sum(axis=1)[active].mean()),1))

strat: PrecomputedAlpha(tag=mom30)
rebalances/yr: 365.0
first active bar: 2019-02-05
per-bar GROSS (active) mean: 1.0
per-bar NET (active) max |sum|: 1.3877787807814457e-16
max single |raw weight| (pre risk-cap): 0.3501
mean #long / #short per active bar: 11.5 / 14.6


In [3]:
# Cell 3 - Costs: three configs (swap vs close+reopen churn vs swap-free)
from backtest.costs import CostModel, CompositeCostModel, Commission, Spread, MarketImpact, LiquidityCap

class GrossHoldingCost(CostModel):
    """Per-bar holding cost on TOTAL gross notional (both long & short legs pay).
    Models FTMO swap (both legs, swap_long=swap_short=-30%) or daily close+reopen churn.
    NOTE: harsher than the engine's ShortBorrow, which only charges the short leg."""
    def __init__(self, annual_pct=None, daily_bps=None, days=365):
        if (annual_pct is None) == (daily_bps is None):
            raise ValueError("pass exactly one of annual_pct / daily_bps")
        self.daily_rate = (annual_pct/days) if annual_pct is not None else (daily_bps/1e4)
        self.kind = "swap" if annual_pct is not None else "churn"
    def trade_cost(self, trades, view): return 0.0
    def holding_cost(self, positions, view):
        return float(positions.abs().sum() * self.daily_rate)

# #006 documented crypto stack
COMM, SPREAD_H, IMPACT_K = 3.25, 5.0, 12      # bps/side, half-spread bps, sqrt-impact k
def common(comm=True):                        # shared rebalance trade-cost (isolates swap-vs-churn)
    c = [Spread(half_bps=SPREAD_H), MarketImpact(k=IMPACT_K, kind="sqrt", adv_lookback=20)]
    if comm: c.insert(0, Commission(bps=COMM))
    return c
liq = LiquidityCap(cap_pct=0.05, adv_lookback=20)   # 5% ADV throttle (engine `liquidity=` arg)

SWEEP = [0, 1, 2, 5, 10]                       # reopen half-spread h (bps) to sweep
configs = {}
configs["hold_through"] = CompositeCostModel(common(True) + [GrossHoldingCost(annual_pct=0.30)])  # pay 30% swap
configs["swap_free"]    = CompositeCostModel(common(True))                                         # ceiling
for h in SWEEP:                                # close+reopen: 365x/yr full-book round-trip, swap=0
    configs[f"reopen_comm_h{h}"]   = CompositeCostModel(common(True)  + [GrossHoldingCost(daily_bps=2*(COMM+h))])
    configs[f"reopen_nocomm_h{h}"] = CompositeCostModel(common(False) + [GrossHoldingCost(daily_bps=2*(0.0+h))])

def hold_bpday(cm):
    return next((m.daily_rate*1e4 for m in cm.models if isinstance(m, GrossHoldingCost)), 0.0)
print("Panel has volume:", panel.has_field("volume"), "| LiqCap cap_pct:", liq.cap_pct)
print(f"{'config':20s} {'hold bp/day':>11s} {'hold %/yr':>9s}  trade comps")
for k, cm in configs.items():
    tc = [type(m).__name__ for m in cm.models if not isinstance(m, GrossHoldingCost)]
    print(f"{k:20s} {hold_bpday(cm):11.2f} {hold_bpday(cm)*365/100:8.1f}%  {tc}")

Panel has volume: True | LiqCap cap_pct: 0.05
config               hold bp/day hold %/yr  trade comps
hold_through                8.22     30.0%  ['Commission', 'Spread', 'MarketImpact']
swap_free                   0.00      0.0%  ['Commission', 'Spread', 'MarketImpact']
reopen_comm_h0              6.50     23.7%  ['Commission', 'Spread', 'MarketImpact']
reopen_nocomm_h0            0.00      0.0%  ['Spread', 'MarketImpact']
reopen_comm_h1              8.50     31.0%  ['Commission', 'Spread', 'MarketImpact']
reopen_nocomm_h1            2.00      7.3%  ['Spread', 'MarketImpact']
reopen_comm_h2             10.50     38.3%  ['Commission', 'Spread', 'MarketImpact']
reopen_nocomm_h2            4.00     14.6%  ['Spread', 'MarketImpact']
reopen_comm_h5             16.50     60.2%  ['Commission', 'Spread', 'MarketImpact']
reopen_nocomm_h5           10.00     36.5%  ['Spread', 'MarketImpact']
reopen_comm_h10            26.50     96.7%  ['Commission', 'Spread', 'MarketImpact']
reopen_nocomm_h10  

In [4]:
# Cell 4 - Engine runs across all 12 cost configs (same alpha, different cost) + net-Sharpe table
from backtest.engine import Engine
from backtest.metrics import compute_metrics

active = weights.abs().sum(axis=1) > 0
start = weights.index[active.argmax()]          # first active bar (skip warmup)
order = (["swap_free", "hold_through"]
         + [f"reopen_comm_h{h}"   for h in SWEEP]
         + [f"reopen_nocomm_h{h}" for h in SWEEP])

results, rows = {}, []
for name in order:
    res = Engine(panel, strat, costs=configs[name], liquidity=liq,
                 initial_capital=1_000_000).run(start=start)
    rep = compute_metrics(res.returns, res.equity_curve, costs=res.costs,
                          trades=res.trades, ann_factor=365)        # crypto = calendar-daily
    results[name] = (res, rep)
    rows.append((name, rep.sharpe, rep.ann_return, rep.max_drawdown, res.costs.sum()))

print(f"run window: {start.date()} -> {panel.dates[-1].date()}  (ann_factor=365)")
print(f"{'config':20s} {'netSharpe':>9s} {'annRet':>8s} {'maxDD':>7s} {'totalCost$':>12s}")
for name, sh, ar, dd, c in rows:
    print(f"{name:20s} {sh:9.2f} {ar:8.1%} {dd:7.1%} {c:12,.0f}")

run window: 2019-02-05 -> 2026-05-21  (ann_factor=365)
config               netSharpe   annRet   maxDD   totalCost$
swap_free                 0.83    22.5%   36.9%    1,113,709
hold_through             -0.14    -8.3%   73.7%    2,719,734
reopen_comm_h0            0.07    -2.4%   64.3%    2,773,485
reopen_comm_h1           -0.17    -9.3%   75.0%    2,705,590
reopen_comm_h2           -0.42   -15.6%   82.5%    2,579,219
reopen_comm_h5           -1.15   -32.2%   94.5%    2,145,921
reopen_comm_h10          -2.35   -52.8%   99.6%    1,669,473
reopen_nocomm_h0          0.88    24.5%   36.4%      751,741
reopen_nocomm_h1          0.65    16.0%   38.3%    1,943,912
reopen_nocomm_h2          0.41     8.2%   40.2%    2,500,729
reopen_nocomm_h5         -0.30   -12.6%   79.2%    2,641,419
reopen_nocomm_h10        -1.51   -39.3%   97.5%    1,962,129
